# 06 — Chunking: Slicing Documents the Right Way

Our vector store works great on 5 short story summaries — each one is small enough to embed as a single document. Real documents aren't that polite: a full screenplay might be 20,000 words, a PDF book 200,000. If we embed one of those as a single document, what do we get back when someone searches it? The *entire* screenplay — even if they asked about one scene in it.

We need to slice documents into smaller pieces first. That's **chunking**, and getting the slice size wrong in either direction breaks retrieval in a different way.


In [ ]:
import os

DATA_DIR = os.path.join("..", "data")
STORIES_DIR = os.path.join(DATA_DIR, "stories")


def load_documents(folder_path: str) -> list[dict]:
    docs = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename)) as f:
                docs.append({"filename": filename, "content": f.read()})
    return docs


story_docs = load_documents(STORIES_DIR)
storyverse_bible = "\n\n".join(d["content"] for d in story_docs)
print(f"'StoryVerse Bible' -- all 5 stories combined: {len(storyverse_bible)} chars")


## Feeling the problem first

If we treat that whole combined blob as one document, a question about Harry Potter's plot retrieves everything, Batman included.


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
bible_embedding = embedder.encode([storyverse_bible])[0]

print("We just embedded", len(storyverse_bible), "characters as ONE vector.")
print("Ask anything -- 'What does Harry learn about his parents?' -- and this entire blob")
print("is what comes back. We'd be sending 900 irrelevant words to answer with 10 relevant ones.")


## What chunking actually is

Chunking splits a long document into smaller, sometimes overlapping, pieces. Each **chunk** gets its own embedding, and search retrieves individual chunks — not whole documents.

Think of a book: it has chapters, paragraphs, sentences. You don't search for the book. You search for the paragraph. The real question is: how big should a chunk be?


## First attempt — just cut every N characters

The simplest thing that could possibly work: slice the text every N characters, no matter what's actually there at the cut point.


In [ ]:
def fixed_size_chunks(text: str, chunk_size: int = 200, overlap: int = 0) -> list[str]:
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(text), step):
        chunks.append(text[i:i + chunk_size])
    return chunks


interstellar_text = next(d["content"] for d in story_docs if d["filename"] == "interstellar.txt")
naive_chunks = fixed_size_chunks(interstellar_text, chunk_size=120, overlap=0)

for i, chunk in enumerate(naive_chunks[:4]):
    print(f"Chunk {i}: {chunk!r}")


Look closely at the boundary between two consecutive chunks above — odds are good one of them cuts off mid-word or mid-sentence. Fixed-size chunking is brutally simple: it doesn't know what a sentence is, so it slices right through them.


## A quick patch — overlap

Before we throw out fixed-size chunking entirely, one cheap patch: if each chunk shares a few characters with the one before it, whatever got sliced off at a boundary at least survives in the neighboring chunk.


In [ ]:
overlapping_chunks = fixed_size_chunks(interstellar_text, chunk_size=120, overlap=30)

print("Chunk 2 (no overlap):  ", repr(naive_chunks[2]))
print("Chunk 2 (with overlap):", repr(overlapping_chunks[2]))


Too little overlap and chunks stay disconnected at the seams. Too much and the same content gets retrieved multiple times, wasting context. A common rule of thumb: **10-20% overlap** relative to chunk size.


## Second attempt — stop ignoring sentences

Overlap patches the symptom; it doesn't stop the cut from happening mid-sentence in the first place. Let's actually fix that: try cutting at *natural* boundaries first — paragraph breaks, then sentence breaks, then word breaks — and only fall back to a hard character cut when nothing natural is available.


In [ ]:
def recursive_split(text: str, chunk_size: int = 300, separators=None) -> list[str]:
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    if len(text) <= chunk_size:
        return [text]

    for sep in separators:
        if sep and sep in text:
            parts = text.split(sep)
            chunks, current = [], ""
            for part in parts:
                candidate = current + sep + part if current else part
                if len(candidate) <= chunk_size:
                    current = candidate
                else:
                    if current:
                        chunks.append(current)
                    current = part
            if current:
                chunks.append(current)
            return chunks

    # Last resort: no separator found, hard-cut by character
    return fixed_size_chunks(text, chunk_size=chunk_size)


smart_chunks = recursive_split(interstellar_text, chunk_size=200)
for i, chunk in enumerate(smart_chunks):
    print(f"Chunk {i} ({len(chunk)} chars): {chunk!r}")


Compare these chunks to the fixed-size ones above — these end at sentence boundaries instead of mid-word. This exact logic — try separators from biggest to smallest, fall back to a hard cut — is what LangChain's `RecursiveCharacterTextSplitter` does internally.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)
lc_chunks = splitter.split_text(interstellar_text)

for i, chunk in enumerate(lc_chunks):
    print(f"Chunk {i} ({len(chunk)} chars): {chunk!r}")


Same idea as our own `recursive_split`, with overlap handling and edge cases already ironed out. Reminder: this is our pure-Python function with a nicer, more battle-tested API — not magic.


## A third attempt — what if we split by meaning?

Sentence boundaries fix the mid-word cuts, but a chunk can still cram two unrelated scenes together if they happen to sit in adjacent sentences. What if, instead of character or sentence boundaries, we split wherever the *topic* actually changes? Embed each sentence, and cut whenever similarity to the previous sentence drops sharply.


In [ ]:
import re
import math


def cosine_similarity(a, b) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x**2 for x in a))
    mag_b = math.sqrt(sum(x**2 for x in b))
    return dot / (mag_a * mag_b)


def semantic_chunk(sentences: list[str], model, threshold: float = 0.6) -> list[str]:
    embeddings = model.encode(sentences)
    chunks, current_chunk = [], [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(embeddings[i - 1], embeddings[i])
        if sim < threshold:  # topic shift detected
            chunks.append(" ".join(current_chunk))
            current_chunk = []
        current_chunk.append(sentences[i])

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks


sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", interstellar_text) if s.strip()]
semantic_chunks = semantic_chunk(sentences, embedder, threshold=0.55)

for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i}: {chunk!r}")


Semantic chunking is more expensive (every sentence needs an embedding) but pays off on documents with clearly distinct sections — Earth scenes vs. space scenes vs. tesseract scenes, in Interstellar's case. For short, single-topic story summaries like ours, the difference is subtle; it matters far more on long, multi-scene documents like a screenplay.


## The Goldilocks experiment

This is the payoff. Three chunk sizes, same question — see which one actually gets retrieved. This time we chunk the full `storyverse_bible` from the very first cell, not just the Interstellar story on its own — a "too large" chunk needs *other stories* next to it to show what noise actually looks like; Interstellar's own text alone is short enough to never get noisy no matter how large the chunk size gets.


In [ ]:
chunk_configs = {"too small": 60, "just right": 200, "too large": 1500}
question = "What does Cooper send to Murph from inside the tesseract?"
question_embedding = embedder.encode([question])[0]

for label, size in chunk_configs.items():
    chunks = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size * 0.15)).split_text(storyverse_bible)
    chunk_embeddings = embedder.encode(chunks)
    scored = sorted(zip(chunks, chunk_embeddings), key=lambda p: cosine_similarity(question_embedding, p[1]), reverse=True)
    best_chunk, _ = scored[0]
    print(f"--- {label} (size={size}, {len(chunks)} chunks) ---")
    print(best_chunk)
    print()


Eyeball the three results: **too small** likely cuts off before the actual answer, or loses the surrounding context needed to make sense of it. **Just right** should capture the full relevant sentence with enough surrounding detail to be useful. **Too large** should now actually pull in a neighboring story's text alongside the real answer, since the chunk boundary no longer lines up with where Interstellar's text starts and ends in the combined bible.

| Chunk size | Retrieved content | Context efficiency |
|---|---|---|
| Too small | Loses meaning at the edges | Cheap, but often wrong |
| Just right | Captures full context, nothing extra | Balanced |
| Too large | Correct, but noisy | Expensive, dilutes attention |

Nothing here says 200 characters is a magic number — the point is that "just right" respects sentence boundaries *and* keeps enough surrounding context, and that's a property you tune per document type, not a constant you hardcode once.


## Attaching metadata

Every chunk should carry a little bookkeeping: which file it came from, its position, so you can cite sources later and debug bad retrievals.


In [ ]:
chunks_with_metadata = [
    {
        "content": chunk,
        "metadata": {
            "source": "interstellar.txt",
            "chunk_index": i,
        },
    }
    for i, chunk in enumerate(RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30).split_text(interstellar_text))
]

for c in chunks_with_metadata[:3]:
    print(c["metadata"], "->", c["content"][:60], "...")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Chunk | One small, searchable slice of a larger document |
| Fixed-size chunking | Cutting every N characters, ignoring sentence structure |
| Recursive character splitting | Cutting at the largest natural boundary that fits (paragraph, then sentence, then word) |
| Overlap | Characters shared between consecutive chunks, so context isn't lost at the seams |
| Semantic chunking | Splitting wherever meaning shifts between sentences, not by a fixed size |

Default rules worth keeping: start with `RecursiveCharacterTextSplitter`, somewhere around 200-500 characters for short content, 10-20% overlap, and always attach metadata.

**Next notebook:** take these chunked, embedded pieces and wire them into a full retrieval pipeline — chunking plus embeddings plus vector search, working together end to end.

**Exercises:**

- Run chunk sizes 50, 100, 200, 500, 1000 on Interstellar's text and watch how the retrieved chunk changes each time.
- Pick a long piece of text of your own (a Wikipedia paragraph works) and find the chunk size that keeps sentences intact.
- Modify `recursive_split` so a chunk never ends mid-sentence, even in the character-level fallback case.
